In [2]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from typing import List

In [3]:
load_dotenv()
EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data
frequency = "weekly" # Frequency of the spot prices, either daily, weekly, or monthly.
series_ids = ["RBRTE", "RWTC"] # ID's of series we're pulling (Brent, WTI)
length = 5000

def series_line(series_ids) -> List[str]: # Build the text used to specify what series we want
    text = [f"&facets[series][]={id}" for id in series_ids]
    return ''.join(text)


URL_BASE = f"https://api.eia.gov/v2/petroleum/pri/spt/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
response = requests.get(url= URL_BASE) # Make request to EIA API
data = response.json()

In [4]:
df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
df.head()

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
4150,1986-01-03,YCUOK,NA,EPCWTI,WTI Crude Oil,PF4,Spot Price FOB,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",25.78,$/BBL
4149,1986-01-10,YCUOK,NA,EPCWTI,WTI Crude Oil,PF4,Spot Price FOB,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",25.99,$/BBL
4148,1986-01-17,YCUOK,NA,EPCWTI,WTI Crude Oil,PF4,Spot Price FOB,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",24.57,$/BBL
4147,1986-01-24,YCUOK,NA,EPCWTI,WTI Crude Oil,PF4,Spot Price FOB,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",20.31,$/BBL
4146,1986-01-31,YCUOK,NA,EPCWTI,WTI Crude Oil,PF4,Spot Price FOB,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",19.69,$/BBL


In [6]:
df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
df = df[['period', 'value', 'series']]
df_wide = df.pivot(index='period', columns = 'series', values= 'value')
df_wide = df_wide.dropna()
df_wide


series,RBRTE,RWTC
period,,
1987-05-15,18.58,19.52
1987-05-22,18.54,19.85
1987-05-29,18.60,19.34
1987-06-05,18.70,19.73
1987-06-12,18.75,19.88
...,...,...
2026-05-15,110.53,105.10
2026-05-22,110.61,105.32
2026-05-29,97.05,93.45


In [7]:
def fetch_eia_spot_prices(series_ids=['RBRTE', 'RWTC'], # ID's of series we're pulling (Brent, WTI)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)


    URL_BASE = f"https://api.eia.gov/v2/petroleum/pri/spt/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide
fetch_eia_spot_prices()

series,RBRTE,RWTC
period,,
1987-05-15,18.58,19.52
1987-05-22,18.54,19.85
1987-05-29,18.60,19.34
1987-06-05,18.70,19.73
1987-06-12,18.75,19.88
...,...,...
2026-05-15,110.53,105.10
2026-05-22,110.61,105.32
2026-05-29,97.05,93.45
